## 1. Before You Begin

MAI-Transcribe-1.5 is a speech recognition model served through the **LLM Speech
API** (public preview). Three things set it apart from MAI-Transcribe-1:

- **43 languages** in multi-lingual mode, with automatic language detection
- **`transcribeStyle`** — readability-optimised output by default, or `verbatim`
  to preserve filler words and disfluencies
- **`phraseList`** — entity biasing toward domain vocabulary

This notebook establishes a **baseline** (transcript, latency, duration, errors —
saved to `output/`) and compares the three MAI-1.5 features against it. For
measured accuracy under controlled noise, see the companion notebook
[mai-transcribe-1.5-noise-benchmark.ipynb](mai-transcribe-1.5-noise-benchmark.ipynb),
which mixes noise at fixed SNR levels and scores WER/CER.

**Pricing:** $0.36 per hour of audio —
[source](https://microsoft.ai/models/mai-transcribe-1-5/)

**Prerequisites**

1. A Microsoft Foundry (AI Services / Speech) resource in a region where LLM
   Speech is available — see
   [Speech service regions](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/regions?tabs=llmspeech).
   See [models/quickstart/](../../../quickstart/README.md) for first-time setup.
2. The environment variables listed in section 2. **No model deployment is
   needed** — you name the model in the request itself.
3. `azure-ai-transcription`, `azure-identity`, `python-dotenv`, `pandas`, and
   `soundfile` installed (section 2 installs them).

**Limits and constraints**
([source: Learn docs](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry))

| Rule | Value |
|---|---|
| Audio formats | WAV, MP3, FLAC |
| Max audio size | 300 MB |
| Diarization | Not supported |
| Prompt-tuning | Not supported |
| `transcribeStyle` / `phraseList` | `mai-transcribe-1.5` only |

The audio in [`data/`](data/) is a set of deliberately hard contact-centre clips —
background rain, room noise, whispering, and far-field speech.


## 2. Set up your environment

The transcription client talks to the **resource** endpoint, not the project
endpoint. `MICROSOFT_FOUNDRY_ENDPOINT` has the form:

```
https://<resource>.services.ai.azure.com/api/projects/<project>
```

Stripping the path suffix gives the base URL the LLM Speech API expects. Set
`AZURE_SPEECH_ENDPOINT` to override this when your Speech resource is separate
from your Foundry project resource.

Authentication uses the API key when one is present, and falls back to
`DefaultAzureCredential` (Entra ID) otherwise — the recommended path for
production.


In [ ]:
%pip install azure-ai-transcription azure-identity python-dotenv pandas soundfile --quiet


In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

from azure.core.credentials import AzureKeyCredential

foundry_endpoint = os.environ.get("MICROSOFT_FOUNDRY_ENDPOINT")
speech_endpoint = os.environ.get("AZURE_SPEECH_ENDPOINT")

if not (speech_endpoint or foundry_endpoint):
    raise EnvironmentError(
        "Set MICROSOFT_FOUNDRY_ENDPOINT (or AZURE_SPEECH_ENDPOINT) before continuing."
    )

if not speech_endpoint:
    parsed = urlparse(foundry_endpoint)
    speech_endpoint = f"{parsed.scheme}://{parsed.netloc}"

API_KEY = os.environ.get("AZURE_SPEECH_API_KEY") or os.environ.get("MICROSOFT_FOUNDRY_API_KEY")
if API_KEY:
    CREDENTIAL = AzureKeyCredential(API_KEY)
    auth = "API key"
else:
    from azure.identity import DefaultAzureCredential

    CREDENTIAL = DefaultAzureCredential()
    auth = "Entra ID (DefaultAzureCredential)"

ENDPOINT = speech_endpoint
MODEL = os.environ.get("MAI_TRANSCRIBE_MODEL", "mai-transcribe-1.5")
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

SUPPORTED_FORMATS = (".wav", ".mp3", ".flac")
audio_files = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in SUPPORTED_FORMATS)
if not audio_files:
    raise FileNotFoundError(
        f"No {'/'.join(SUPPORTED_FORMATS)} audio found in {DATA_DIR.resolve()}"
    )

print(f"Endpoint : {ENDPOINT}")
print(f"Model    : {MODEL}")
print(f"Auth     : {auth}")
print(f"Output   : {OUT_DIR.resolve()}")
print(f"Clips    : {len(audio_files)}")
for f in audio_files:
    print(f"  - {f.name}")

## 3. Configure the client

`TranscriptionClient.transcribe()` takes a `TranscriptionContent` that pairs the
audio bytes with a `TranscriptionOptions` definition. Everything MAI-specific
lives in `enhanced_mode`:

| Option | Where it goes | Effect |
|---|---|---|
| `model` | `enhanced_mode` | Selects `mai-transcribe-1.5` |
| `transcribeStyle` | `enhanced_mode` | `verbatim` keeps disfluencies; omit for readability |
| `locales` | `TranscriptionOptions` | Forces a single language instead of auto-detection |
| `phrase_list` | `TranscriptionOptions` | Biases recognition toward supplied phrases |

`model` and `transcribeStyle` aren't typed attributes on `EnhancedModeProperties`
yet, so they're set as dictionary keys — the SDK models are mutable mappings and
serialize the extra keys straight through to the REST payload.

Two helpers below do the work for the rest of the notebook:

- `transcribe(...)` — one call, raises on failure
- `run(...)` — wraps `transcribe` and returns a **record**: transcript, audio
  duration, wall-clock latency, real-time factor, and the error string if the
  call failed. Records go straight into a DataFrame, so a failed clip doesn't
  abort a batch.


In [ ]:
import time

from azure.ai.transcription import TranscriptionClient
from azure.ai.transcription.models import (
    EnhancedModeProperties,
    PhraseListProperties,
    TranscriptionContent,
    TranscriptionOptions,
)

client = TranscriptionClient(endpoint=ENDPOINT, credential=CREDENTIAL)


def transcribe(
    audio_path,
    *,
    locales: list[str] | None = None,
    style: str | None = None,
    phrases: list[str] | None = None,
    biasing_weight: float | None = None,
):
    """Transcribe one file with MAI-Transcribe-1.5 and return the result object."""
    enhanced = EnhancedModeProperties(task="transcribe")
    enhanced["model"] = MODEL
    if style:
        enhanced["transcribeStyle"] = style

    opts: dict = {"enhanced_mode": enhanced}
    if locales:
        opts["locales"] = locales
    if phrases:
        opts["phrase_list"] = PhraseListProperties(
            phrases=phrases,
            biasing_weight=biasing_weight if biasing_weight is not None else 1.0,
        )

    with open(audio_path, "rb") as audio:
        content = TranscriptionContent(definition=TranscriptionOptions(**opts), audio=audio)
        return client.transcribe(content)


def run(audio_path, *, label: str, **kwargs) -> dict:
    """Transcribe and return a record row: transcript, timings, and any error."""
    audio_path = Path(audio_path)
    record = {
        "clip": audio_path.name,
        "format": audio_path.suffix.lstrip("."),
        "run": label,
        "requested_locales": ",".join(kwargs.get("locales") or []) or "auto",
        "detected_locales": "",
        "style": kwargs.get("style") or "default",
        "phrase_list": len(kwargs.get("phrases") or []),
        "transcript": "",
        "mean_confidence": None,
        "audio_seconds": None,
        "latency_seconds": None,
        "rtf": None,
        "error": None,
    }

    started = time.perf_counter()
    try:
        result = transcribe(audio_path, **kwargs)
    except Exception as exc:  # a failed clip must not abort the batch
        record["error"] = f"{type(exc).__name__}: {exc}"
        record["latency_seconds"] = round(time.perf_counter() - started, 2)
        return record

    latency = time.perf_counter() - started
    audio_seconds = result.duration_milliseconds / 1000
    confidences = [p.confidence for p in result.phrases if p.confidence is not None]
    record["transcript"] = result.combined_phrases[0].text
    record["detected_locales"] = ",".join(sorted({p.locale for p in result.phrases if p.locale}))
    record["mean_confidence"] = round(sum(confidences) / len(confidences), 3) if confidences else None
    record["audio_seconds"] = round(audio_seconds, 2)
    record["latency_seconds"] = round(latency, 2)
    record["rtf"] = round(latency / audio_seconds, 3) if audio_seconds else None
    return record


def show(result, title: str = "", detail: bool = False) -> None:
    """Print the combined transcript, plus per-phrase timings when detail=True."""
    if title:
        print(f"--- {title} ---")
    print(result.combined_phrases[0].text)
    print(f"[audio duration: {result.duration_milliseconds / 1000:.1f}s]")
    if detail:
        print("\nPhrases:")
        for phrase in result.phrases:
            locale = phrase.locale or "?"
            print(
                f"  [{phrase.offset_milliseconds:>6} ms] ({locale}, "
                f"conf {phrase.confidence:.2f}) {phrase.text}"
            )
    print()

## 4. Run a baseline transcription

The baseline answers two questions and leaves a record you can diff against
later runs:

1. Does the model handle the audio at all — WAV, MP3, or FLAC, clean or noisy?
2. Does **automatic language detection** cost you anything versus pinning
   `locales=["en"]`?

The clips in `data/` are all MP3, so the clean one is transcoded to 16-bit WAV
and FLAC first — all three accepted formats go through the same call path, and
the `format` column shows whether the container moves the numbers. (It shouldn't:
WAV and FLAC carry identical samples here, FLAC just at roughly half the size.)

Each call is logged as a row — transcript, audio duration, wall-clock latency,
real-time factor, and any error — and the table is written to
`output/baseline_results.csv`. Detection ambiguity usually shows up on short or
noisy audio, so the auto-detect / pinned-locale pair is repeated on a
rain-and-room-noise clip.


In [ ]:
import pandas as pd
import soundfile as sf

clean_clip = DATA_DIR / "normal - on call agent building.mp3"
noisy_clip = DATA_DIR / "rainy weather and on call agent.mp3"

# Transcode the clean clip so the baseline exercises all three accepted formats.
FORMAT_DIR = OUT_DIR / "formats"
FORMAT_DIR.mkdir(exist_ok=True)

samples, sample_rate = sf.read(clean_clip, dtype="float32", always_2d=True)
mono = samples.mean(axis=1)
clean_wav = FORMAT_DIR / f"{clean_clip.stem}.wav"
clean_flac = FORMAT_DIR / f"{clean_clip.stem}.flac"
sf.write(clean_wav, mono, sample_rate, subtype="PCM_16")
sf.write(clean_flac, mono, sample_rate, subtype="PCM_16")
print(f"Transcoded {clean_clip.name} → {clean_wav.name}, {clean_flac.name} @ {sample_rate} Hz mono")

baseline_runs = [
    (clean_clip, "clean mp3 · auto-detect", {}),
    (clean_clip, "clean mp3 · locale=en", {"locales": ["en"]}),
    (clean_wav, "clean wav · locale=en", {"locales": ["en"]}),
    (clean_flac, "clean flac · locale=en", {"locales": ["en"]}),
    (noisy_clip, "noisy mp3 · auto-detect", {}),
    (noisy_clip, "noisy mp3 · locale=en", {"locales": ["en"]}),
]

baseline = pd.DataFrame([run(path, label=label, **kwargs) for path, label, kwargs in baseline_runs])
baseline.to_csv(OUT_DIR / "baseline_results.csv", index=False)
print(f"Saved → {OUT_DIR / 'baseline_results.csv'}")

display(
    baseline[
        [
            "clip",
            "format",
            "run",
            "requested_locales",
            "detected_locales",
            "mean_confidence",
            "audio_seconds",
            "latency_seconds",
            "rtf",
            "error",
        ]
    ]
)

for _, row in baseline.iterrows():
    print(f"\n--- {row['clip']} · {row['run']} ---")
    print(row["error"] or row["transcript"])

# Auto-detect vs. pinned locale, side by side.
print("\n--- language detection ---")
for _, row in baseline.iterrows():
    print(
        f"  {row['run']:<26} requested={row['requested_locales']:<5} "
        f"detected={row['detected_locales'] or '—':<8} conf={row['mean_confidence']}"
    )


## 5. Compare readability and verbatim transcribe styles

`transcribeStyle` is new in `mai-transcribe-1.5`:

- **default (omitted)** — a readability-optimised transcript: punctuation
  normalised, filler words and stammers cleaned up. Use for summaries, captions,
  and anything a human reads.
- **`verbatim`** — preserves the original spoken content including "um", false
  starts, and repetitions. Use for compliance review, QA scoring, and speech
  analytics where disfluencies are signal, not noise.

Run both styles over the same clip and diff them.


In [ ]:
import difflib

style_clip = DATA_DIR / "noisy room - oncall agent.mp3"

readable = transcribe(style_clip, locales=["en"])
verbatim = transcribe(style_clip, locales=["en"], style="verbatim")

show(readable, title="Default (readability-optimised)")
show(verbatim, title="transcribeStyle = verbatim")

print("--- word-level diff (readability → verbatim) ---")
diff = difflib.ndiff(
    readable.combined_phrases[0].text.split(),
    verbatim.combined_phrases[0].text.split(),
)
print(" ".join(token for token in diff if token[0] in "+-"))

## 6. Bias recognition toward domain vocabulary

Generic speech models mis-hear names, SKUs, and acronyms that never appeared in
training data. `phraseList` implements **entity biasing**: you hand the model a
short list of expected terms, and it weights recognition toward them.

Good candidates: product and brand names, agent and customer names, ticket ID
formats, drug names, internal acronyms.

Keep the list tight — every extra phrase dilutes the bias. `biasing_weight`
(0.0–2.0) controls how strongly the list is applied; start at 1.0 and raise it
only if terms are still being missed.


In [ ]:
bias_clip = DATA_DIR / "rain + specific content.mp3"

# Replace these with the vocabulary your own audio actually contains.
DOMAIN_PHRASES = ["Contoso", "Fabrikam", "Northwind Traders", "SLA", "escalation tier 2"]

unbiased = transcribe(bias_clip, locales=["en"])
biased = transcribe(bias_clip, locales=["en"], phrases=DOMAIN_PHRASES, biasing_weight=1.5)

show(unbiased, title="No phrase list")
show(biased, title=f"phraseList = {DOMAIN_PHRASES}")

print("--- phrases recovered by biasing ---")
for phrase in DOMAIN_PHRASES:
    in_base = phrase.lower() in unbiased.combined_phrases[0].text.lower()
    in_biased = phrase.lower() in biased.combined_phrases[0].text.lower()
    print(f"  {phrase:<20} baseline={in_base!s:<5} biased={in_biased}")

## 7. Your Turn to Explore

Push on the three knobs you just used:

1. **Widen the baseline** — feed every clip in `data/` through `run()` and sort
   `output/baseline_results.csv` by `mean_confidence`. Which recording condition
   (whispering, far-field, heavy rain) degrades recognition most?
2. **Tune the biasing weight** — sweep `biasing_weight` from 0.5 to 2.0 on the
   same phrase list. Where does biasing start hallucinating your terms into
   audio that never contained them?
3. **Style the output for a downstream task** — transcribe a call in both styles
   and feed each into a summarisation prompt. Does verbatim input change the
   summary, or just cost more tokens?

Then quantify it: the
[noise-robustness notebook](mai-transcribe-1.5-noise-benchmark.ipynb) turns these
impressions into WER and CER against controlled SNR levels.


In [ ]:
# Your experiments here


## 8. Summary

You used MAI-Transcribe-1.5 through the LLM Speech API to:

- Establish a baseline over clean and noisy audio in all three accepted formats
  (MP3, WAV, FLAC) — transcript, detected locale, mean confidence, audio
  duration, latency, real-time factor, and errors, logged to
  `output/baseline_results.csv`
- Compare automatic language detection against a pinned locale
- Switch between readability-optimised and `verbatim` transcripts with
  `transcribeStyle`
- Bias recognition toward domain vocabulary with `phraseList` and
  `biasing_weight`

**When to reach for this model:** multilingual transcription where you need
either clean human-readable output or faithful verbatim output from the same
service, and where domain vocabulary matters. Reach elsewhere when you need
speaker diarization or prompt-tuning — neither is supported here.

**Next:** [mai-transcribe-1.5-noise-benchmark.ipynb](mai-transcribe-1.5-noise-benchmark.ipynb)
— mix café/traffic/music/white noise at 20/10/5/0 dB SNR and score WER and CER
against the references in `data/reference/`.

**References**

- [MAI-Transcribe in Azure Speech (preview)](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-transcribe?context=%2Fazure%2Ffoundry%2Fcontext%2Fcontext&pivots=ai-foundry) — `enhancedMode`, `transcribeStyle`, `phraseList`, language support, and limitations
- [LLM Speech for speech transcription and translation](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/llm-speech?tabs=new-foundry%2Cwindows&pivots=programming-language-python) — auth, the `azure-ai-transcription` SDK, and the response shape
- [Introducing MAI-Transcribe-1.5](https://microsoft.ai/news/mai-transcribe-1-5more-accurate-context-aware-and-built-for-production/) — release announcement
- [MAI-Transcribe-1.5 model page](https://microsoft.ai/models/mai-transcribe-1-5/) — benchmarks against MAI-Transcribe-1 and pricing per audio hour
- [MAI-Transcribe-1.5 model card (PDF)](https://microsoft.ai/pdf/MAI-Transcribe-1.5-Model-Card.PDF) — intended uses, evaluation methodology, responsible-AI considerations
- [Audio / Speech primer](../../../../docs/primers/audio-speech.md) · [Glossary](../../../../docs/GLOSSARY.md)
